# PydanticAI 라이브러리의 Agent 클래스를 이용한 LLM 출력 구조화

## 환경 설정

In [1]:
# 필요한 모듈 / 클래스 / 함수 임포트
import os
import json
import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from langgraph.graph import StateGraph, END

In [2]:
# API Key 가져오기
file_path='.env'
output=load_dotenv(file_path)
print(output)

True


## 그래프 상태 정의

In [3]:
# BaseModel 클래스 상속 -> 컴스텀 자료형 AgentState 생성
class AgentState(BaseModel):
    # 사용자의 요청 사항 저장
    request: str = ""
    # LLM이 출력한 원시 텍스트(구조화되지 않은 출력)
    raw_response: str = ""
    # 구조화된 결과 데이터
    structured_data: dict = {}
    # 에러 발생 시 기록할 메시지
    error_message: str = ""

## [제1부] 실패 시나리오

### 노드 함수 정의

In [4]:
# [노드 1] 분석 노드 -> 사용자 요청 분석 -> "항목", "수량" 등이 포함된 무질서한 텍스트 생성 -> 다음 노드: 처리 실패

async def unsafe_research_node(state: AgentState) -> dict:
    print("[Node 1] 제조 현장 자재 수량 데이터 분석 중...")

    # 실제 LLM 대신 시뮬레이션 실행 (비구조화된 텍스트 응답)
    llm_bad_output = """
    네, 요청하신 자재 재고 분석 결과입니다!
    ```json
    {
    'item_name': '고강도 볼트M8',
    'current_stock': '수량 부족 (재고 15개)'
    }
    ```
    추가 조치가 필요하면 말씀하세요.
    """

    # 함수 실행의 결과(return) -> dict 자료형
    return {'raw_response': llm_bad_output}

In [5]:
# [노드 2] 가공 처리 노드: 데이터를 넘겨받아 '정수(int)' 연산을 처리
async def unsafe_process_node(state: AgentState) -> dict:
    print("[Node 2] 분석된 데이터를 바탕으로 자동 발주 수량 계산 시작...")
    print('-'*80)

    # 상태 확인: 항목1=값1 항목2=값2 ... 
    print(f'state: \n{state}')
    print('-'*80)

    # 앞 단계(분석 노드) LLM이 출력한 원시 텍스트 추출
    raw_text = state.raw_response

    # LLM이 출력한 원시 텍스트 -> 가공(json 문자열 -> dict 자료형) -> 현재 수량 추출 -> 주문 수량 파악  
    try:
        # LLM이 출력한 원시 텍스트 -> json 문자열로 변환: '{"키":값}'
        bad_json_str = raw_text.split("```json")[1].split("```")[0].strip()
        clean_json_str = bad_json_str.replace("'", '"')

        # dict 자료형으로 변환 -> json.loads() 호출
        data = json.loads(clean_json_str)

        # dict 자료형으로 변환 -> 'current_stock' 키의 값 추출
        # [치명적 문제 발생] 숫자가 와야 하는 'current_stock'에 "수량 부족 (재고 15개)"라는 텍스트가 들어와서 
        # 형변환 에러 발생
        stock_count = int(data["current_stock"])
        print(f"현재 재고가 {stock_count}개 확인되었습니다.")

        print('-'*80)

        # 안전 수량(100개) 기준 -> current_stock -> 미달 시 부족 수량 계산 -> 자동 주문 처리
        default_qty = 100
        order_qty = default_qty - stock_count
        
        # 함수 실행 결과(return) -> dict 자료형
        return {"structured_data": {"item": data['item_name'], "order_qty": order_qty}}

    except Exception as e:
        error_msg=f"에러 발생! 에러 내용: {e}"
        return {'error_message': error_msg}

### 그래프 생성 및 컴파일 (실패 케이스 워크플로우)

In [6]:
# StateGraph 클래스 호출 -> 그래프 생성
unsafe_workflow = StateGraph(AgentState)

# 그래프: 노드 추가
unsafe_workflow.add_node("research", unsafe_research_node)
unsafe_workflow.add_node("process", unsafe_process_node)

# 그래프: 시작점 설정
unsafe_workflow.set_entry_point("research")

# 그래프: 엣지 추가
unsafe_workflow.add_edge("research", "process")
unsafe_workflow.add_edge("process", END)

# 컴파일 
unsafe_agent = unsafe_workflow.compile()

### 실행 및 결과 확인

In [7]:
# main 함수 정의
async def main():
    # 사용자 요청 -> 초기 상태 설정 -> dict 자료형 {"messages":[HumanMessage]}
    initial_state = {
        "request": "현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치 요망"
    }

    # agent -> 비동기 처리 방식 실행 -> 최종 상태 생성
    unsafe_state = await unsafe_agent.ainvoke(initial_state)

    # 최종 상태 -> dict 자료형 -> 에러 메시지 추출 및 출력
    if unsafe_state.get("error_message"):
        print(f"\n[결과 알림] {unsafe_state['error_message']}")
        print("한 노드의 비구조화된 출력이 에이전트 전체 파이프라인을 중단시켰습니다.")

# main 함수 실행
await main()

[Node 1] 제조 현장 자재 수량 데이터 분석 중...
[Node 2] 분석된 데이터를 바탕으로 자동 발주 수량 계산 시작...
--------------------------------------------------------------------------------
state: 
request='현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치 요망' raw_response="\n    네, 요청하신 자재 재고 분석 결과입니다!\n    ```json\n    {\n    'item_name': '고강도 볼트M8',\n    'current_stock': '수량 부족 (재고 15개)'\n    }\n    ```\n    추가 조치가 필요하면 말씀하세요.\n    " structured_data={} error_message=''
--------------------------------------------------------------------------------

[결과 알림] 에러 발생! 에러 내용: invalid literal for int() with base 10: '수량 부족 (재고 15개)'
한 노드의 비구조화된 출력이 에이전트 전체 파이프라인을 중단시켰습니다.


## [제2부] PydanticAI Agent 구조화 된 출력 방식 (성공 시나리오)

### 구조화된 출력을 위한 자료형 클래스 정의

In [8]:
# LLM이 지켜야 할 출력 형식을 지정 -> '상태'처럼 작성 -> BaseModel 클래스 상속 (유효성 검사 기능)
# [노드 1] 분석 노드 -> LLM 출력 -> item_name="볼트" current_stock=15  
class InventoryAnalysis(BaseModel):
    item_name: str = Field(description="자재 품목 명칭")
    current_stock: int = Field(description="현재 창고에 남아있는 정확한 재고 수량 (숫자만 허용)") 

### PydanticAI Agent 정의

In [9]:
# 기존 LLM 대체 

# Agent 클래스 호출 -> LLM 출력 관리 도구 생성
pydantic_agent = Agent(
    # llm = ChatOpenAI(model="gpt-5-nano") 대신
    model='openai:gpt-5-nano',
    # llm이 지켜야 할 출력 형식
    output_type=InventoryAnalysis,
    system_prompt="당신은 제조 공장의 자재 관리 전문가입니다. 텍스트 문맥을 분석하여 재고 수량을 정확한 정수(int)로만 추출하세요."
)

### 노드 함수 정의

In [10]:
# [노드 1] PydanticAI Agent 클래스를 사용하는 안전한 분석 노드 
async def safe_research_node(state: AgentState) -> dict:
    print("[Node 1] PydanticAI 기반 자재 데이터 분석 중 ...")
    print('-'*80)

    # 상태 -> 사용자 요청 사항 추출
    user_request = state.request
    print(f"요청 사항: {user_request}")

    print('-'*80)

    # 사용자 프롬프트 작성
    user_prompt = f"다음 요청에서 품목 명칭과 현재 재고 수량을 정수로 추출해줘: {user_request}"

    # pydantic_agent(llm 출력 관리 도구) -> 비동기 실행 함수 -> Agent 클래스 -> 전용 함수 run() 호출
    agent_result = await pydantic_agent.run(user_prompt=user_prompt)
    print(f'pydantic_agent 실행 결과: \n{agent_result}')
    print('-'*80)

    # pydantic_agent 실행 결과 추출 -> InventoryAnalysis 자료형 -> 구조화된 출력 생성 -> dict() 자료형(X)
    agent_output = agent_result.output
    print(f'pydantic_agent 실행 결과: {agent_output}')
    print('-'*80)

    # pydantic_agent 실행 결과 -> dict 자료형 -> 상태 업데이트
    # BaseModel 클래스 -> 전용 함수 -> model_dump() 함수 -> 커스텀 자료형 -> dict 자료형
    # 결과: item_name="볼트" current_stock=15 -> {'item_name':'고강도 볼트M8', 'current_stock'=15}
    agent_output_dict = agent_output.model_dump()

    # 함수 실행의 결과(return) -> dict 자료형
    return {"structured_data": agent_output_dict} 

In [11]:
# [노드2] 노드 1의 결과를 전달받아서 부족한 수량 연산
async def safe_process_node(state: AgentState) -> dict:
    print("[Node 2] 구조화된 데이터를 바탕으로 부족한 수량 연산 시작...")
    print('-'*80)

    # 상태 -> structured_data = {'item_name':'고강도 볼트M8', 'current_stock'=15} -> 구조화된 데이터 추출 
    data = state.structured_data 

    # 현재 재고 수량 추출: data -> {'item_name':'고강도 볼트M8', 'current_stock'=15}
    stock_count = data["current_stock"]
    print(f"현재 재고가 정확히 '{stock_count}'개 확인되었습니다.")
    print('-'*80)

    # 주문 수량 연산
    default_qty = 100
    order_qty = default_qty - stock_count
    print(f"[자동 발주 확정] 품목: {data['item_name']} | 발주 필요 수량: {order_qty}개")
    print('-'*80)

    # 함수의 실행의 결과(return) -> dict 자료형
    return {"error_message": "없음 (비동기 파이프라인 성공)"}

### 그래프 생성 및 컴파일 

In [12]:
# StateGraph 클래스 호출 -> 상태 기반 그래프 생성
safe_workflow = StateGraph(AgentState)

# 그래프: 노드 추가
safe_workflow.add_node("safe_research", safe_research_node)
safe_workflow.add_node("safe_process", safe_process_node)

# 그래프: 시작점 설정
safe_workflow.set_entry_point("safe_research")

# 그래프: 엣지 추가
safe_workflow.add_edge("safe_research", "safe_process")
safe_workflow.add_edge("safe_process", END)

# 컴파일
safe_agent = safe_workflow.compile()

### 실행 및 결과 확인

In [13]:
# main 함수 정의
async def main():

    # 초기 상태 설정
    initial_state = {
        "request": "현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치 요망"
    }

    print("에이전트 시스템 가동 ...")
    print('-'*80)

    # agent 비동기 실행 -> await, ainvoke() -> 최종 상태 생성 -> dict 자료형
    final_state = await safe_agent.ainvoke(initial_state)

    # 최종 상태 -> 에러 메시지 확인(추출)
    error_msg = final_state['error_message']
    print(f"[최종 결과(error)] {error_msg}")

# main 함수 실행
await main()


에이전트 시스템 가동 ...
--------------------------------------------------------------------------------
[Node 1] PydanticAI 기반 자재 데이터 분석 중 ...
--------------------------------------------------------------------------------
요청 사항: 현재 창고 확인 결과 고강도 볼트M8의 수량이 부족하여 경고 알림(실재고는 현재 15개 남음). 조속히 조치 요망
--------------------------------------------------------------------------------


c:\Users\playdata2\miniconda3\envs\agent\Lib\site-packages\pydantic_ai\models\__init__.py:1984: DeprecatedModelWarning: Model 'openai:gpt-5-nano' is deprecated by its provider, with a retirement date of 2026-12-11. It recommends 'gpt-6-luna' instead. See https://developers.openai.com/api/docs/deprecations#2026-06-11-gpt-5-and-o3-model-deprecations
  warnings.warn(


pydantic_agent 실행 결과: 
AgentRunResult(output=InventoryAnalysis(item_name='고강도 볼트M8', current_stock=15))
--------------------------------------------------------------------------------
pydantic_agent 실행 결과: item_name='고강도 볼트M8' current_stock=15
--------------------------------------------------------------------------------
[Node 2] 구조화된 데이터를 바탕으로 부족한 수량 연산 시작...
--------------------------------------------------------------------------------
현재 재고가 정확히 '15'개 확인되었습니다.
--------------------------------------------------------------------------------
[자동 발주 확정] 품목: 고강도 볼트M8 | 발주 필요 수량: 85개
--------------------------------------------------------------------------------
[최종 결과(error)] 없음 (비동기 파이프라인 성공)
